# 原子分辨 STM 图像多特征分割 — SAM 3（自动阈值）

使用 SAM 3 对原子分辨表面图像进行多特征自动分割，包括：

- **CDW 超结构 / 表面重构**（十字星形亮斑）

- **台阶边缘**（亮条带）

- **点缺陷 / 空位**（暗斑点）

**自动确定阈值**，无需手动调参。




## 1. 环境准备





In [ ]:
import sys, os, numpy as np
from pathlib import Path
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import torch

# 中文字体（Windows 兼容）
for _f in matplotlib.font_manager.findSystemFonts():
    if any(k in _f for k in ("NotoSansCJK", "NotoSerifCJK", "msyh", "SimHei", "Microsoft YaHei")):
        matplotlib.rcParams["font.family"] = matplotlib.font_manager.FontProperties(fname=_f).get_name()
        break
matplotlib.rcParams["axes.unicode_minus"] = False

# ── 项目路径 ──────────────────────────────────────────────
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "lumen").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the Lumen repository.")

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
SAM3_ROOT = str(PROJECT_ROOT / "SEM zero-shot")
sys.path.insert(0, SAM3_ROOT)

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"Project: {PROJECT_ROOT}")





## 2. 构建 SAM 3





In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
CHECKPOINT_PATH = str(PROJECT_ROOT / "checkpoints" / "sam3" / "sam3.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Building SAM 3...")

model = build_sam3_image_model(
    checkpoint_path=CHECKPOINT_PATH, device=DEVICE,
    eval_mode=True, enable_segmentation=True, enable_inst_interactivity=False,
)

print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

# ── 修复 BFloat16/Float dtype 不匹配 ─────────────────────

if DEVICE == "cuda":
    import sam3.model.vitdet as _vitdet
    _original_addmm_act = _vitdet.addmm_act
    def _patched_addmm_act(activation, linear, mat1):
        orig_dtype = mat1.dtype
        return _original_addmm_act(activation, linear, mat1).to(orig_dtype)
    _vitdet.addmm_act = _patched_addmm_act
    print("Applied addmm_act dtype patch")

# 关键: threshold=0 获取全部 200 个预测
processor = Sam3Processor(model, resolution=1008, device=DEVICE, confidence_threshold=0.0)
print("Processor ready")





## 3. 加载图像目录

批量处理 `png-modulation` 下全部 20 张 FeTe STM 图像。




In [ ]:
# ============================================================
# Image directory and output directory
# ============================================================
IMAGE_DIR = PROJECT_ROOT / "data" / "phase1_unlabeled" / "STM" / "Fete" / "PNG"
OUT_DIR = PROJECT_ROOT / "SEM zero-shot" / "output-sam3-phase1-fete-balanced"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CROP_TOP = 50
CROP_BOTTOM = 0
CROP_LEFT = 0
CROP_RIGHT = 0

image_paths = sorted(IMAGE_DIR.glob("*.png"))
print(f"Found {len(image_paths)} images:")
for path in image_paths:
    print(f"  {path.name}")


def load_and_crop(path):
    """Load one RGB image and apply the configured crop."""
    image = Image.open(path).convert("RGB")
    if max(image.size) > 1008:
        image.thumbnail((1008, 1008), Image.LANCZOS)
    if CROP_TOP > 0 or CROP_BOTTOM > 0 or CROP_LEFT > 0 or CROP_RIGHT > 0:
        width, height = image.size
        image = image.crop((CROP_LEFT, CROP_TOP, width - CROP_RIGHT, height - CROP_BOTTOM))
    return image


## 4. SAM3 candidate layer and balanced preliminary layer

SAM3 text prompts are kept unchanged. SAM3 is used as a **candidate generator**, not as a calibrated physical classifier.

- `raw`: high-recall candidate aggregation, retained for comparison.
- `balanced`: uses a small score-ranked candidate budget and transparent component-area ranges calibrated for this 20-image FeTe set.
- No cross-class overlap or uncertainty layer is rendered or exported. Each prompt is shown independently.

The `sqrt2_modulation_region` result remains candidate-only: without a local Fourier/wave-vector check it must not be interpreted as a final ?2 physical count.


In [ ]:
import cv2
import time

# ── Per-image scan calibration (GPT image2 analysis) ─────────────
# scan_width_nm: lateral scan size derived from scale bar
# Default 40 nm for uncalibrated FeTe images.
def select_candidates(scores, config):
    """Keep a score-ranked SAM3 proposal prefix and record the score elbow."""
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    if scores.size == 0:
        return np.empty(0, dtype=np.int64), {"raw": 0, "selected": 0}

    order = np.argsort(scores)[::-1]
    search = min(int(config.get("elbow_window", 30)), scores.size - 1)
    if search <= 0:
        elbow_rank = 1
        keep_n = 1
    else:
        sorted_scores = scores[order]
        drops = sorted_scores[:search] - sorted_scores[1 : search + 1]
        elbow_rank = int(np.argmax(drops)) + 1
        keep_n = elbow_rank + int(config.get("elbow_shift", 0))

    keep_n = max(1, min(keep_n, int(config["max_candidates"]), scores.size))
    selected = order[:keep_n].copy()  # avoid PyTorch negative-stride indexing
    return selected, {
        "raw": int(scores.size),
        "elbow_rank": int(elbow_rank),
        "selected": int(keep_n),
        "threshold": float(scores[selected[-1]]),
    }


def multi_prompt_segment(processor, image_pil, prompt_configs, verbose=True):
    """Run SAM3 once per unchanged text prompt."""
    results = {}
    for config in prompt_configs:
        state = processor.set_image(image_pil)
        output = processor.set_text_prompt(prompt=config["prompt"], state=state)
        scores = output["scores"].detach().cpu().numpy()
        selected, audit = select_candidates(scores, config)

        selected_tensor = torch.as_tensor(
            selected, dtype=torch.long, device=output["masks"].device
        )
        masks = output["masks"][selected_tensor]
        boxes = output["boxes"][selected_tensor]
        kept_scores = scores[selected]
        audit["prompt"] = config["prompt"]
        if verbose:
            print(
                f"    {config['name']}: raw={audit['raw']}, "
                f"elbow={audit['elbow_rank']}, selected={audit['selected']}"
            )

        results[config["name"]] = {
            "masks": masks,
            "boxes": boxes,
            "scores": kept_scores,
            "config": config,
            "audit": audit,
        }
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return results


def to_binary_mask(mask, height, width):
    """Resize one SAM3 mask and use a single explicit foreground rule."""
    if hasattr(mask, "detach"):
        mask = mask.detach().cpu().numpy()
    mask = np.squeeze(np.asarray(mask))
    if mask.shape != (height, width):
        mask = cv2.resize(
            mask.astype(np.float32), (width, height), interpolation=cv2.INTER_NEAREST
        )
    if mask.dtype == np.bool_:
        return mask.astype(np.uint8)
    if float(mask.min()) < 0:
        return (mask > 0).astype(np.uint8)
    return (mask >= 0.5).astype(np.uint8)


def mask_aspect_ratio(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) < 3:
        return float("inf")
    covariance = np.cov(np.stack([xs, ys]))
    eigenvalues = np.sort(np.linalg.eigvalsh(covariance))
    return float(np.sqrt(max(eigenvalues[-1], 1e-6) / max(eigenvalues[0], 1e-6)))


def local_contrast_z(gray, mask):
    """Mean in-mask contrast against a small surrounding ring."""
    kernel = np.ones((7, 7), dtype=np.uint8)
    outer = cv2.dilate(mask, kernel, iterations=1)
    ring = (outer > 0) & (mask == 0)
    inside = mask > 0
    if not np.any(ring) or not np.any(inside):
        return 0.0
    return float((gray[inside].mean() - gray[ring].mean()) / max(float(gray.std()), 1e-6))


def mask_iou(first, second):
    intersection = int(np.logical_and(first > 0, second > 0).sum())
    union = int(np.logical_or(first > 0, second > 0).sum())
    return intersection / union if union else 0.0


def nms_candidates(candidates, iou_threshold):
    """Remove only near-identical same-prompt proposals."""
    kept = []
    for candidate in sorted(candidates, key=lambda item: item["sam_score"], reverse=True):
        if all(mask_iou(candidate["mask"], prior["mask"]) < iou_threshold for prior in kept):
            kept.append(candidate)
    return kept


def build_candidates(all_results, image_rgb, candidate_limits, nms_iou_threshold):
    """Convert selected SAM3 proposals to binary candidates without hard filtering."""
    height, width = image_rgb.shape[:2]
    total_area = height * width
    all_candidates, audit = {}, {}

    for name, result in all_results.items():
        config = result["config"]
        limit = min(int(candidate_limits[name]), len(result["scores"]))
        min_area = int(np.ceil(config["min_area_ratio"] * total_area))
        max_area = int(np.floor(config["max_area_ratio"] * total_area))
        valid, rejected_area = [], 0

        for mask, score in zip(result["masks"][:limit], result["scores"][:limit]):
            binary = to_binary_mask(mask, height, width)
            area = int(cv2.countNonZero(binary))
            if not min_area <= area <= max_area:
                rejected_area += 1
                continue
            valid.append({
                "mask": binary,
                "sam_score": float(score),
                "area": area,
                "area_ratio": area / total_area,
            })

        kept = nms_candidates(valid, nms_iou_threshold)
        all_candidates[name] = kept
        audit[name] = {
            **result["audit"],
            "profile_limit": int(limit),
            "area_kept": len(valid),
            "nms_kept": len(kept),
            "rejected_area": int(rejected_area),
        }
    return all_candidates, audit


def masks_from_candidates(candidates_by_class, min_area=10):
    """Union each prompt's proposals, then split the union into components."""
    feature_masks = {}
    for name, candidates in candidates_by_class.items():
        masks = [candidate["mask"] for candidate in candidates]
        if not masks:
            feature_masks[name] = []
            continue
        composite = np.maximum.reduce(masks).astype(np.uint8)
        labels_count, labels = cv2.connectedComponents(composite, connectivity=8)
        feature_masks[name] = [
            (labels == label).astype(np.uint8)
            for label in range(1, labels_count)
            if cv2.countNonZero((labels == label).astype(np.uint8)) >= min_area
        ]
    return feature_masks


def filter_feature_masks(feature_masks, image_rgb, component_rules):
    """Apply transparent size rules after connected-component aggregation.

    Pixel rules are used intentionally: this 20-image set spans different STM
    fields of view, and unverified per-image nanometre calibration would create
    false precision. The effective minimum is the stricter of a fixed pixel
    floor and an image-area fraction.
    """
    height, width = image_rgb.shape[:2]
    total_area = height * width
    filtered, audit = {}, {}

    for name, masks in feature_masks.items():
        rule = component_rules[name]
        min_area = max(
            int(rule["min_area_px"]),
            int(np.ceil(float(rule.get("min_area_ratio", 0.0)) * total_area)),
        )
        max_area = int(np.floor(float(rule["max_area_ratio"]) * total_area))
        kept, rejected = [], {"too_small": 0, "too_large": 0}
        for mask in masks:
            area = int(cv2.countNonZero(mask))
            if area < min_area:
                rejected["too_small"] += 1
                continue
            if area > max_area:
                rejected["too_large"] += 1
                continue
            kept.append(mask)
        filtered[name] = kept
        audit[name] = {
            "raw_components": len(masks),
            "kept_components": len(kept),
            "effective_min_area_px": min_area,
            "effective_max_area_px": max_area,
            "rejected": rejected,
            "rule": rule,
        }
    return filtered, audit


def extract_features(feature_masks, image_rgb):
    """Export one record per connected component for every prompt."""
    height, width = image_rgb.shape[:2]
    total_area = height * width
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    all_features = {}
    for name, masks in feature_masks.items():
        records = []
        for index, mask in enumerate(masks):
            area = int(cv2.countNonZero(mask))
            moments = cv2.moments(mask, binaryImage=True)
            if not area or not moments["m00"]:
                continue
            records.append({
                "index": index,
                "cx": moments["m10"] / moments["m00"],
                "cy": moments["m01"] / moments["m00"],
                "area_px": area,
                "area_ratio": area / total_area,
                "equivalent_radius_px": float(np.sqrt(area / np.pi)),
                "aspect_ratio": mask_aspect_ratio(mask),
                "local_contrast_z": local_contrast_z(gray, mask),
                "intensity_mean": float(cv2.mean(gray, mask=mask)[0]),
            })
        all_features[name] = records
    return all_features


def union_of_masks(masks, shape):
    if not masks:
        return np.zeros(shape, dtype=np.uint8)
    return np.maximum.reduce(masks).astype(np.uint8)


_LEGEND_LABELS = {
    "bright_defect": "Bright defect",
    "modulation_region": "Modulation region",
    "dark_defect": "Dark defect",
    "sqrt2_modulation_region": "Sqrt2 candidate",
}


def draw_legend(image, entries):
    font = cv2.FONT_HERSHEY_SIMPLEX
    margin, swatch, line_height = 10, 14, 22
    labels = [f"{entry['label']} ({entry['count']})" for entry in entries]
    text_width = max(cv2.getTextSize(label, font, 0.50, 1)[0][0] for label in labels)
    height, width = image.shape[:2]
    box_width = margin + swatch + 8 + text_width + margin
    box_height = margin + line_height * len(entries) + margin // 2
    x0, y0 = width - box_width - margin, height - box_height - margin
    shaded = image.copy()
    cv2.rectangle(shaded, (x0, y0), (width - margin, height - margin), (0, 0, 0), -1)
    cv2.addWeighted(shaded, 0.55, image, 0.45, 0, dst=image)
    for index, (entry, label) in enumerate(zip(entries, labels)):
        y = y0 + margin + index * line_height
        cv2.rectangle(image, (x0 + margin, y), (x0 + margin + swatch, y + swatch), entry["color"], -1)
        cv2.rectangle(image, (x0 + margin, y), (x0 + margin + swatch, y + swatch), (255, 255, 255), 1)
        cv2.putText(image, label, (x0 + margin + swatch + 8, y + swatch - 2), font, 0.50, (255, 255, 255), 1, cv2.LINE_AA)


def render_overlay(image_rgb, feature_masks, prompt_configs, alpha=0.30):
    """Render compact English labels to avoid CJK font availability issues."""
    visual = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    entries = []
    for config in prompt_configs:
        masks = feature_masks.get(config["name"], [])
        if masks:
            union = union_of_masks(masks, image_rgb.shape[:2])
            colored = visual.copy()
            colored[union > 0] = config["color"]
            cv2.addWeighted(colored, alpha, visual, 1 - alpha, 0, dst=visual)
            contours, _ = cv2.findContours(union, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(visual, contours, -1, config["color"], 1)
        entries.append({"label": _LEGEND_LABELS[config["key"]], "color": config["color"], "count": len(masks)})
    draw_legend(visual, entries)
    return visual


def save_overlay(image_rgb, feature_masks, prompt_configs, save_path):
    visual = render_overlay(image_rgb, feature_masks, prompt_configs)
    ok, encoded = cv2.imencode(".png", visual, [cv2.IMWRITE_PNG_COMPRESSION, 1])
    if not ok:
        raise RuntimeError(f"Could not encode {save_path}")
    Path(save_path).write_bytes(encoded.tobytes())


# Original prompt text is intentionally unchanged.
prompt_configs = [
    {"name": "bright defect", "key": "bright_defect", "prompt": "a bright cross dot", "elbow_shift": 200, "max_candidates": 200, "color": (64, 255, 64), "min_area_ratio": 0.0, "max_area_ratio": 0.005},
    {"name": "modulation region", "key": "modulation_region", "prompt": "a bright stripe across image", "elbow_shift": 10, "max_candidates": 200, "color": (255, 255, 64), "min_area_ratio": 0.02, "max_area_ratio": 0.50},
    {"name": "dark defect", "key": "dark_defect", "prompt": "a dark dot", "elbow_shift": 200, "max_candidates": 200, "color": (64, 64, 255), "min_area_ratio": 0.0, "max_area_ratio": 0.01},
    {"name": "sqrt2_modulation_region", "key": "sqrt2_modulation_region", "prompt": "checkerboard atomic lattice with alternating bright atoms", "elbow_shift": 10, "max_candidates": 200, "color": (255, 64, 255), "min_area_ratio": 0.02, "max_area_ratio": 0.50},
]

# "raw" reproduces high-recall aggregation. "balanced" uses a small
# score-ranked proposal budget because repeated lattice crosses otherwise fill
# the 200-candidate defect tails. No brittle contrast or shape gate is used.
POSTPROCESS_PROFILE = "balanced"  # choose "raw" to reproduce the old behavior
FILTER_PROFILES = {
    "raw": {
        "candidate_limits": {config["name"]: 200 for config in prompt_configs},
        "nms_iou_threshold": 1.10,
        "component_rules": {config["name"]: {"min_area_px": 10, "min_area_ratio": 0.0, "max_area_ratio": 1.0} for config in prompt_configs},
    },
    "balanced": {
        "candidate_limits": {"bright defect": 200, "modulation region": 12, "dark defect": 200, "sqrt2_modulation_region": 12},
        "nms_iou_threshold": 0.80,
        "component_rules": {
            "bright defect": {"min_area_px": 30, "min_area_ratio": 0.0, "max_area_ratio": 0.0050},
            "modulation region": {"min_area_px": 100, "min_area_ratio": 0.008, "max_area_ratio": 0.30},
            "dark defect": {"min_area_px": 20, "min_area_ratio": 0.0, "max_area_ratio": 0.0015},
            "sqrt2_modulation_region": {"min_area_px": 100, "min_area_ratio": 0.015, "max_area_ratio": 0.12},
        },
    },
}
CANDIDATE_ONLY_CLASSES = {"sqrt2_modulation_region"}

print(f"Defined {len(prompt_configs)} unchanged prompts; active profile: {POSTPROCESS_PROFILE}")


## 5. Batch inference and two-layer export

For every image, SAM3 inference runs once. The notebook then derives both the raw candidate aggregation and the active preliminary profile from the same proposals.

`*_balanced_overlay.png` is the default visual output. Set `SAVE_RAW_OVERLAY = True` only when you need all 20 raw comparison images; it increases disk writes. No overlap/uncertainty mask is produced.


In [ ]:
import csv
import json

SAVE_BALANCED_OVERLAY = True
SAVE_RAW_OVERLAY = False
SAVE_JSON_CSV = True
SAVE_RAW_COMPONENTS = False

if POSTPROCESS_PROFILE not in FILTER_PROFILES:
    raise ValueError(f"Unknown POSTPROCESS_PROFILE: {POSTPROCESS_PROFILE}")

raw_profile = FILTER_PROFILES["raw"]
active_profile = FILTER_PROFILES[POSTPROCESS_PROFILE]
all_summaries = []
all_batch_results = {}
total_t0 = time.time()

for image_index, image_path in enumerate(image_paths, start=1):
    stem = image_path.stem
    print(f"\n[{image_index}/{len(image_paths)}] {stem}")
    image_pil = load_and_crop(image_path)
    image_rgb = np.array(image_pil)

    infer_t0 = time.time()
    raw_results = multi_prompt_segment(processor, image_pil, prompt_configs, verbose=False)
    inference_seconds = time.time() - infer_t0

    post_t0 = time.time()
    raw_candidates, raw_candidate_audit = build_candidates(
        raw_results,
        image_rgb,
        raw_profile["candidate_limits"],
        raw_profile["nms_iou_threshold"],
    )
    raw_feature_masks = masks_from_candidates(raw_candidates)
    raw_features = extract_features(raw_feature_masks, image_rgb)

    if POSTPROCESS_PROFILE == "raw":
        active_candidates = raw_candidates
        active_candidate_audit = raw_candidate_audit
    else:
        active_candidates, active_candidate_audit = build_candidates(
            raw_results,
            image_rgb,
            active_profile["candidate_limits"],
            active_profile["nms_iou_threshold"],
        )
    preliminary_masks = masks_from_candidates(active_candidates)
    feature_masks, component_audit = filter_feature_masks(
        preliminary_masks, image_rgb, active_profile["component_rules"]
    )
    all_features = extract_features(feature_masks, image_rgb)
    postprocess_seconds = time.time() - post_t0

    save_t0 = time.time()
    if SAVE_BALANCED_OVERLAY:
        save_overlay(
            image_rgb,
            feature_masks,
            prompt_configs,
            OUT_DIR / f"{stem}_balanced_overlay.png",
        )
    if SAVE_RAW_OVERLAY:
        save_overlay(
            image_rgb,
            raw_feature_masks,
            prompt_configs,
            OUT_DIR / f"{stem}_raw_overlay.png",
        )
    if SAVE_JSON_CSV:
        for config in prompt_configs:
            name, key = config["name"], config["key"]
            records = all_features[name]
            with open(OUT_DIR / f"{stem}_{key}.json", "w", encoding="utf-8") as handle:
                json.dump(records, handle, indent=2, ensure_ascii=False)
            if records:
                with open(OUT_DIR / f"{stem}_{key}.csv", "w", newline="", encoding="utf-8") as handle:
                    writer = csv.DictWriter(handle, fieldnames=records[0].keys())
                    writer.writeheader()
                    writer.writerows(records)
            if SAVE_RAW_COMPONENTS:
                with open(OUT_DIR / f"{stem}_{key}_raw_components.json", "w", encoding="utf-8") as handle:
                    json.dump(raw_features[name], handle, indent=2, ensure_ascii=False)

        audit_payload = {
            "profile": POSTPROCESS_PROFILE,
            "raw_candidate_audit": raw_candidate_audit,
            "active_candidate_audit": active_candidate_audit,
            "component_audit": component_audit,
            "candidate_only_classes": sorted(CANDIDATE_ONLY_CLASSES),
        }
        with open(OUT_DIR / f"{stem}_audit.json", "w", encoding="utf-8") as handle:
            json.dump(audit_payload, handle, indent=2, ensure_ascii=False)
    save_seconds = time.time() - save_t0

    summary = {
        "image": stem,
        "size": [int(image_rgb.shape[1]), int(image_rgb.shape[0])],
        "profile": POSTPROCESS_PROFILE,
        "n_raw_components": {name: len(records) for name, records in raw_features.items()},
        "n_features": {name: len(records) for name, records in all_features.items()},
        "candidate_audit": active_candidate_audit,
        "component_audit": component_audit,
        "candidate_only_classes": sorted(CANDIDATE_ONLY_CLASSES),
    }
    all_summaries.append(summary)
    all_batch_results[stem] = {
        "image_rgb": image_rgb,
        "raw_feature_masks": raw_feature_masks,
        "feature_masks": feature_masks,
        "raw_features": raw_features,
        "features": all_features,
        "raw_audit": raw_candidate_audit,
        "audit": active_candidate_audit,
        "component_audit": component_audit,
    }

    counts = ", ".join(
        f"{config['key']}={len(raw_features[config['name']])}->{len(all_features[config['name']])}"
        for config in prompt_configs
    )
    print(
        f"  raw->balanced: {counts}; infer={inference_seconds:.1f}s; "
        f"post={postprocess_seconds:.2f}s; save={save_seconds:.2f}s"
    )

if SAVE_JSON_CSV:
    with open(OUT_DIR / "batch_summary.json", "w", encoding="utf-8") as handle:
        json.dump(all_summaries, handle, indent=2, ensure_ascii=False)

total_seconds = time.time() - total_t0
print(f"\nDone: {len(all_summaries)} images in {total_seconds:.1f}s")
print(f"Output: {OUT_DIR}")


## 6. Raw-to-balanced count summary

The table reports `raw ? balanced` components. These are image-processing components, not automatically validated physical defect counts. In particular, ?2 is candidate-only until a local Fourier validation stage is added.


In [ ]:
print(f"{'Image':<14}  Raw -> balanced component counts")
print("-" * 108)
for summary in all_summaries:
    counts = " | ".join(
        f"{config['key']}: {summary['n_raw_components'].get(config['name'], 0)}->{summary['n_features'].get(config['name'], 0)}"
        for config in prompt_configs
    )
    print(f"{summary['image']:<14}  {counts}")

print("-" * 108)
for config in prompt_configs:
    name = config["name"]
    raw_values = [summary["n_raw_components"].get(name, 0) for summary in all_summaries]
    balanced_values = [summary["n_features"].get(name, 0) for summary in all_summaries]
    label = "candidate only" if name in CANDIDATE_ONLY_CLASSES else "preliminary"
    print(
        f"{config['key']:<28} mean raw->balanced: "
        f"{np.mean(raw_values):.1f}->{np.mean(balanced_values):.1f} ({label})"
    )


## 7. Three-image calibration view

Inspect the same three reference images before trusting the active profile. The middle panel is the high-recall raw candidate union; the right panel is the active balanced preliminary output.


In [ ]:
CALIBRATION_IMAGES = ["FeTe_0009", "FeTe_0012", "FeTe_0019"]

missing = [name for name in CALIBRATION_IMAGES if name not in all_batch_results]
if missing:
    raise RuntimeError("Run the batch-inference cell before calibration display: " + ", ".join(missing))

fig, axes = plt.subplots(len(CALIBRATION_IMAGES), 3, figsize=(15, 5 * len(CALIBRATION_IMAGES)))
for row, stem in enumerate(CALIBRATION_IMAGES):
    result = all_batch_results[stem]
    image_rgb = result["image_rgb"]
    raw_bgr = render_overlay(image_rgb, result["raw_feature_masks"], prompt_configs)
    balanced_bgr = render_overlay(image_rgb, result["feature_masks"], prompt_configs)

    axes[row, 0].imshow(image_rgb)
    axes[row, 0].set_title(f"{stem}: source")
    axes[row, 1].imshow(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(f"{stem}: raw candidates")
    axes[row, 2].imshow(cv2.cvtColor(balanced_bgr, cv2.COLOR_BGR2RGB))
    axes[row, 2].set_title(f"{stem}: balanced preliminary")
    for axis in axes[row]:
        axis.axis("off")

plt.tight_layout()
plt.show()


## 8. All balanced results

Display all 20 active-profile overlays without sampling. Only balanced overlays are collected here, so raw comparison images do not duplicate the grid.


In [ ]:
SAVE_FIGURE = False
FIGURE_PATH = OUT_DIR / "all_balanced_overlays_summary.png"
all_overlay_paths = sorted(OUT_DIR.glob("*_balanced_overlay.png"))

n = len(all_overlay_paths)
cols = 4
rows = max(1, (n + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
axes = np.asarray(axes).reshape(-1)
for axis, path in zip(axes, all_overlay_paths):
    axis.imshow(Image.open(path))
    axis.set_title(path.stem.replace("_balanced_overlay", ""), fontsize=11)
    axis.axis("off")
for axis in axes[n:]:
    axis.axis("off")

plt.suptitle(f"SAM3 balanced preliminary segmentation: all {n} images", fontsize=15, y=1.0)
plt.tight_layout()
if SAVE_FIGURE:
    fig.savefig(str(FIGURE_PATH), dpi=150, bbox_inches="tight")
    print(f"Saved: {FIGURE_PATH}")
plt.show()


## 9. Interpretation note

The notebook separates proposal generation from preliminary filtering. The balanced profile is calibrated to the candidate-count and component-size distributions of this 20-image FeTe set. It does not use a cross-class uncertainty layer. Do not use the exported `sqrt2_modulation_region` component count as a physical ?2 count until a local Fourier/wave-vector validation method is added.
